# 04 — Tek Tank Derinlemesine Analiz

**IST_001, Tank 1** (Motorin, 23000 lt, manifoldlu) üzerinde gerçek sistemdeki
"istasyon seç → tank detayları" akışını simüle ediyoruz.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name != 'eda':
    ROOT = ROOT / 'eda'
sys.path.insert(0, str(ROOT))

from utils.data_loader import load_all, filter_tank, merge_ue1t_inventory
from utils.plots import setup_style

setup_style()
dfs = load_all()

IST, TANK = 'IST_001', 1
tank_info = filter_tank(dfs['tanks'], IST, TANK)
daily = filter_tank(dfs['daily'], IST, TANK).sort_values('tarih')
ue1t = filter_tank(dfs['ue1t_30min'], IST, TANK).sort_values('saat_1')
tx = filter_tank(dfs['transactions'], IST, TANK).sort_values('satis_zamani')
inv = filter_tank(dfs['inventory_30min'], IST, TANK).sort_values('envanter_tarihi')
deliv = filter_tank(dfs['deliveries'], IST, TANK)

print('Tank bilgisi:')
display(tank_info.T)
print(f'\nSatır sayıları: daily={len(daily)}, ue1t={len(ue1t)}, tx={len(tx)}, inv={len(inv)}, deliv={len(deliv)}')

In [ ]:
# Günlük zaman serisi — 4 panel
fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)
daily.plot(x='tarih', y='satis', ax=axes[0], legend=False, color='steelblue')
axes[0].set_ylabel('Satış (lt)')
daily.plot(x='tarih', y='fark', ax=axes[1], legend=False, color='coral')
axes[1].axhline(0, color='k', lw=0.5)
axes[1].set_ylabel('Fark (lt)')
daily.plot(x='tarih', y='kapanis', ax=axes[2], legend=False, color='green')
axes[2].axhline(tank_info.kapasite.iloc[0]*0.04, color='r', ls='--', label='dead stock ~4%')
axes[2].set_ylabel('Kapanış stok')
daily.plot(x='tarih', y='alarm', ax=axes[3], legend=False, drawstyle='steps-post', color='red')
axes[3].set_ylabel('Alarm (0/1)')
axes[3].set_xlabel('Tarih')
fig.suptitle(f'{IST} Tank {TANK} — 90 günlük özet', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Tek gün 30 dk detay — en yüksek |fark| günü
gun = daily.loc[daily['fark'].abs().idxmax(), 'tarih']
u_gun = ue1t[ue1t.saat_1.dt.normalize() == gun]
print(f'Seçilen gün: {gun.date()} | günlük fark: {daily[daily.tarih==gun].fark.values[0]:.2f} lt')

fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)
u_gun.plot(x='saat_1', y='donem_sonu_stok', ax=axes[0], legend=False)
axes[0].set_ylabel('Stok (lt)')
u_gun.plot(x='saat_1', y='pompa_satis', ax=axes[1], legend=False, color='orange')
axes[1].set_ylabel('30dk satış')
u_gun.plot(x='saat_1', y='kayip_kazanc', ax=axes[2], legend=False, color='red')
axes[2].axhline(0, color='k', lw=0.5)
axes[2].set_ylabel('Kayıp/Kazanç')
axes[2].set_xlabel('Saat')
fig.suptitle(f'{IST} T{TANK} — {gun.date()} UE1T detay')
plt.tight_layout()
plt.show()

In [ ]:
# Saatlik satış profili (tüm dönem)
tx2 = tx.copy()
tx2['saat'] = tx2['satis_zamani'].dt.hour + tx2['satis_zamani'].dt.minute/60
profil = tx2.groupby(tx2['satis_zamani'].dt.floor('h'))['litre'].sum()
profil.index = profil.index.hour

hourly = tx2.groupby(tx2['satis_zamani'].dt.hour)['litre'].sum()
fig, ax = plt.subplots(figsize=(12, 4))
hourly.plot(kind='bar', ax=ax, color='steelblue', edgecolor='k')
ax.set_xlabel('Saat')
ax.set_ylabel('Toplam litre')
ax.set_title(f'{IST} T{TANK} — saatlik satış profili (90 gün)')
plt.tight_layout()
plt.show()

In [ ]:
# UE1T + Envanter join
merged = merge_ue1t_inventory(ue1t, inv)
print('Join sonrası satır:', len(merged), '| null sicaklik:', merged['sicaklik_inv'].isna().sum() if 'sicaklik_inv' in merged.columns else merged.filter(like='sicaklik').isna().sum().sum())

fig, ax1 = plt.subplots(figsize=(14, 4))
ax1.plot(merged['saat_1'], merged['donem_sonu_stok'], label='Stok (UE1T)', color='steelblue')
ax1.set_ylabel('Stok (lt)')
ax2 = ax1.twinx()
ax2.plot(merged['saat_1'], merged['sicaklik_ue1t'], label='Sıcaklık', color='orange', alpha=0.7)
ax2.set_ylabel('Sıcaklık (°C)')
ax1.set_title(f'{IST} T{TANK} — stok ve sıcaklık')
fig.tight_layout()
plt.show()

In [ ]:
# Dolum olayları
if len(deliv):
    display(deliv[['dolum_baslangic','dolum_net','dolum_oncesi_hacim','dolum_sonrasi_hacim','sicaklik']].head(10))
    fig, ax = plt.subplots(figsize=(12, 3))
    ax.scatter(deliv['dolum_baslangic'], deliv['dolum_net'], s=60, alpha=0.7)
    ax.set_xlabel('Dolum zamanı')
    ax.set_ylabel('Dolum net (lt)')
    ax.set_title(f'{IST} T{TANK} — dolum olayları')
    plt.show()

## Sonuç

Tek tank üzerinde günlük özet → 30 dk detay → tekil satış → envanter akışını gördük.

Sonraki: tüm tanklar için günlük alarm analizi (`05_gunluk_alarm_ve_fark.ipynb`).